# Decision Tree 

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv('final_input.csv')
df


,sample-name,AB_used,f__Enterobacteriaceae,g__Bacteroides,g__Prevotella,g__Faecalibacterium,g__Roseburia,g__Blautia,g__Alistipes,g__Bifidobacterium,...,f__Rhizobiaceae,g__Alicyclobacillus,f__Staphylococcaceae,f__Methylobacteriaceae,g__Mycoplana,g__Sharpea,f__Solirubrobacterales,f__Paenibacillaceae,g__Brachyspira,g__Tetrathiobacter
0,ERR10359977,1,0.000459,0.309954,0.004648,0.206954,0.017226,0.026215,0.041723,0.008930,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,ERR10360021,1,0.000346,0.200071,0.000798,0.170797,0.129203,0.033302,0.007864,0.022578,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,ERR10360103,1,0.001446,0.406406,0.000000,0.000890,0.015237,0.003225,0.038817,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,ERR10360111,1,0.000000,0.440678,0.000000,0.000000,0.050847,0.000000,0.084746,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,ERR10360131,1,0.000000,0.034177,0.141442,0.081260,0.029994,0.037478,0.008156,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1265,ERR9744223,0,0.010509,0.147732,0.002293,0.103519,0.083266,0.010432,0.035653,0.002751,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1266,ERR9744224,0,0.002569,0.198376,0.059092,0.147935,0.015750,0.055975,0.006856,0.000198,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1267,ERR9744232,0,0.007122,0.083921,0.295755,0.072515,0.024767,0.004148,0.034116,0.002754,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1268,ERR9744241,0,0.001719,0.103305,0.001553,0.107325,0.070090,0.032882,0.023456,0.000499,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
X = df.drop(['AB_used', 'sample-name'], axis=1)
y = df['AB_used']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
mean_value = X_train.mean().mean()
imputer = SimpleImputer(strategy='constant', fill_value=mean_value)



pipeline = Pipeline([
    ('imputer', imputer), 
    ('scaler', StandardScaler()), 
    ('dt', DecisionTreeClassifier(random_state=42)) 
])

param_grid = {
    'dt__max_depth': [3, 5, 7], 
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', verbose=1, error_score='raise')
grid_search.fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)

Fitting 5 folds for each of 3 candidates, totalling 15 fits
Best Parameters: {'dt__max_depth': 5}
Best Score: 0.8335174252523329


In [4]:
y_pred = grid_search.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Improved Accuracy: {accuracy}')

y_pred_proba = grid_search.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print("AUC:", auc)


y_pred = grid_search.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()


specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)

print("Specificity:", specificity)
print("Sensitivity:", sensitivity)


Improved Accuracy: 0.8398950131233596
AUC: 0.8724486973837049
Specificity: 0.8212290502793296
Sensitivity: 0.8564356435643564


In [5]:
from sklearn.metrics import precision_score, recall_score, f1_score
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


Precision: 0.8439024390243902
Recall: 0.8564356435643564
F1 Score: 0.8501228501228502


In [7]:
import numpy as np
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Initialize lists to store the bootstrap results
bootstrap_aucs = []
bootstrap_accuracies = []
bootstrap_precisions = []
bootstrap_recalls = []
bootstrap_f1s = []
bootstrap_specificities = []
n_bootstraps = 1000

for _ in range(n_bootstraps):
    # Bootstrap sample the data indices
    indices = resample(np.arange(len(y_test)), replace=True)
    y_test_sample = y_test.iloc[indices]
    X_test_sample = X_test.iloc[indices]
    
    # Predict and compute metrics
    y_pred_sample = grid_search.predict(X_test_sample)
    y_pred_proba_sample = grid_search.predict_proba(X_test_sample)[:, 1]
    
    # Compute metrics
    bootstrap_auc = roc_auc_score(y_test_sample, y_pred_proba_sample)
    bootstrap_accuracy = accuracy_score(y_test_sample, y_pred_sample)
    bootstrap_precision = precision_score(y_test_sample, y_pred_sample)
    bootstrap_recall = recall_score(y_test_sample, y_pred_sample)
    bootstrap_f1 = f1_score(y_test_sample, y_pred_sample)
    
    # Compute specificity
    tn, fp, fn, tp = confusion_matrix(y_test_sample, y_pred_sample).ravel()
    bootstrap_specificity = tn / (tn + fp)
    
    # Append metrics to lists
    bootstrap_aucs.append(bootstrap_auc)
    bootstrap_accuracies.append(bootstrap_accuracy)
    bootstrap_precisions.append(bootstrap_precision)
    bootstrap_recalls.append(bootstrap_recall)
    bootstrap_f1s.append(bootstrap_f1)
    bootstrap_specificities.append(bootstrap_specificity)

# Calculate the 95% confidence intervals
def calculate_ci(metric_list):
    ci_lower = np.percentile(metric_list, 2.5)
    ci_upper = np.percentile(metric_list, 97.5)
    return ci_lower, ci_upper

auc_ci = calculate_ci(bootstrap_aucs)
accuracy_ci = calculate_ci(bootstrap_accuracies)
precision_ci = calculate_ci(bootstrap_precisions)
recall_ci = calculate_ci(bootstrap_recalls)
f1_ci = calculate_ci(bootstrap_f1s)
specificity_ci = calculate_ci(bootstrap_specificities)

print(f"Bootstrap 95% CI for AUC: {auc_ci[0]:.3f} to {auc_ci[1]:.3f}")
print(f"Bootstrap 95% CI for Accuracy: {accuracy_ci[0]:.3f} to {accuracy_ci[1]:.3f}")
print(f"Bootstrap 95% CI for Precision: {precision_ci[0]:.3f} to {precision_ci[1]:.3f}")
print(f"Bootstrap 95% CI for Recall (Sensitivity): {recall_ci[0]:.3f} to {recall_ci[1]:.3f}")
print(f"Bootstrap 95% CI for F1 Score: {f1_ci[0]:.3f} to {f1_ci[1]:.3f}")
print(f"Bootstrap 95% CI for Specificity: {specificity_ci[0]:.3f} to {specificity_ci[1]:.3f}")


Bootstrap 95% CI for AUC: 0.833 to 0.909
Bootstrap 95% CI for Accuracy: 0.806 to 0.874
Bootstrap 95% CI for Precision: 0.792 to 0.889
Bootstrap 95% CI for Recall (Sensitivity): 0.803 to 0.904
Bootstrap 95% CI for F1 Score: 0.812 to 0.885
Bootstrap 95% CI for Specificity: 0.767 to 0.873


In [8]:
import numpy as np
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score

bootstrap_aucs = []
n_bootstraps = 1000

for _ in range(n_bootstraps):
    # Bootstrap sample the data indices
    indices = resample(np.arange(len(y_test)), replace=True)
    y_test_sample = y_test.iloc[indices]
    X_test_sample = X_test.iloc[indices]
    y_pred_proba_sample = grid_search.predict_proba(X_test_sample)[:, 1]
    bootstrap_auc = roc_auc_score(y_test_sample, y_pred_proba_sample)
    bootstrap_aucs.append(bootstrap_auc)

ci_lower = np.percentile(bootstrap_aucs, 2.5)
ci_upper = np.percentile(bootstrap_aucs, 97.5)

print(f"Bootstrap 95% CI for AUC: {ci_lower:.3f} to {ci_upper:.3f}")

original_auc = roc_auc_score(y_test, grid_search.predict_proba(X_test)[:, 1])

#permutation test to calculate p-value
n_permutations = 10000
perm_aucs = []

for _ in range(n_permutations):
    y_test_permuted = np.random.permutation(y_test)
    perm_auc = roc_auc_score(y_test_permuted, grid_search.predict_proba(X_test)[:, 1])
    perm_aucs.append(perm_auc)

perm_aucs = np.array(perm_aucs)
p_value = np.mean(perm_aucs >= original_auc)

print(f"Permutation test p-value for AUC: {p_value:.3f}")


Bootstrap 95% CI for AUC: 0.832 to 0.912
Permutation test p-value for AUC: 0.000
